# Phase 2: Intelligent Agent and Search Algorithms
### STEP#1: DEFINE AGENT AND STATE SPACE
 1. Initial State: The starting symptom ('itching').
 2. Goal State: The target symptom ('fatigue').
 3. Actions: Transitioning to a connected neighbor in the symptom graph.
 4. Cost: 1 for uninformed search; Severity Score for UCS and A*.
 5. Solution: A sequence (path) of symptoms from start to goal.

In [46]:
import pandas as pd
from collections import deque

df = pd.read_csv(r"C:\Users\qande\Desktop\AI_Project\AI_Project\data\dataset.csv")
severity_df = pd.read_csv(r"C:\Users\qande\Desktop\AI_Project\AI_Project\data\Symptom-severity.csv")
for col in df.columns:
    if col != 'Disease':
        df[col] = df[col].str.strip()

print("Data loaded and cleaned!")

Data loaded and cleaned!


In [47]:
severity_df.columns = severity_df.columns.str.strip()
severity_df['Symptom'] = severity_df['Symptom'].str.strip()
severity_dict = dict(zip(severity_df['Symptom'], severity_df['weight']))
print(f"severity_dict built with {len(severity_dict)} entries.")

severity_dict built with 132 entries.


In [48]:
def build_graph(df):
    symptom_cols = [col for col in df.columns if col.startswith('Symptom')]
    graph = {} 
    for _, row in df.iterrows():
        symptoms = [row[col] for col in symptom_cols if pd.notna(row[col])] 
        for s in symptoms:
            if s not in graph:
                graph[s] = set()
            for other in symptoms:
                if other != s:
                    graph[s].add(other)                
    return {k: list(v) for k, v in graph.items()}

symptom_graph = build_graph(df) # graph created
print(f"Graph built with {len(symptom_graph)} unique symptoms.")

Graph built with 131 unique symptoms.


In [49]:
class AIAgent:
    def __init__(self, symptom_graph):
        self.graph = symptom_graph

    def perceive(self, state):
        return self.graph.get(state, [])

    def act(self, action):
        return action

    def goal_test(self, current_state, target_goal):
        return current_state == target_goal

    def get_cost(self, state1, state2):
        return 1

agent = AIAgent(symptom_graph)
print("AI-Agent is ready!")

AI-Agent is ready!


### STEP#2: UNINFORMED SERACH:

In [51]:
# BFS
def breadth_first_search(agent, start_node, goal_node):
    queue = deque([[start_node]])    
    visited = set() 
    visited.add(start_node)
    nodes_explored = 0
    while queue:
        current_path = queue.popleft()   
        last_symptom = current_path[-1] 
        nodes_explored += 1
        if agent.goal_test(last_symptom, goal_node):
            return current_path, nodes_explored      
        for neighbor in agent.perceive(last_symptom):
            if neighbor not in visited:
                visited.add(neighbor)
                new_path = list(current_path)
                new_path.append(neighbor)
                queue.append(new_path)            
    return None, nodes_explored

start_symptom = 'itching'
target_symptom = 'fatigue'

path, count = breadth_first_search(agent, start_symptom, target_symptom)
print(f"BFS")
print(f"Path Found: {path}")
print(f"Nodes Explored: {count}")

BFS
Path Found: ['itching', 'fatigue']
Nodes Explored: 17


In [52]:
# DFS
def depth_first_search(agent, start_node, goal_node):
    stack = [[start_node]]
    visited = set()  
    nodes_explored = 0
    
    print(f"Starting DFS search from '{start_node}' to find '{goal_node}'...")

    while stack:
        current_path = stack.pop()   
        last_symptom = current_path[-1]
        
        if last_symptom in visited:  
            continue
          
        visited.add(last_symptom)
        nodes_explored += 1
        # Goal Test
        if agent.goal_test(last_symptom, goal_node):
            return current_path, nodes_explored

        neighbors = agent.perceive(last_symptom) 
        
        for neighbor in neighbors:
            if neighbor not in visited:
                new_path = list(current_path) 
                new_path.append(neighbor)
                stack.append(new_path)
             
    return None, nodes_explored

start_symptom = 'itching'
target_symptom = 'fatigue'

dfs_path, dfs_count = depth_first_search(agent, start_symptom, target_symptom)

print(f"DFS")
print(f"Path Found: {dfs_path}")
print(f"Nodes Explored: {dfs_count}")

Starting DFS search from 'itching' to find 'fatigue'...
DFS
Path Found: ['itching', 'yellowing_of_eyes', 'muscle_pain', 'diarrhoea', 'abdominal_pain', 'coma', 'fatigue']
Nodes Explored: 7


In [53]:
#DLS
def depth_limited_search(agent, start_node, goal_node, limit):
    stack = [( [start_node], 0 )]
    visited = {} 
    nodes_explored = 0
    
    print(f"Starting DLS (Limit={limit}) from '{start_node}' to '{goal_node}'...")

    while stack:
        path, depth = stack.pop()
        current_symptom = path[-1]
        
        nodes_explored += 1
        
        if agent.goal_test(current_symptom, goal_node): 
            return path, nodes_explored

        if depth < limit: 
            for neighbor in agent.perceive(current_symptom): 
                if neighbor not in visited or visited[neighbor] > depth + 1: 
                    visited[neighbor] = depth + 1
                    new_path = list(path)
                    new_path.append(neighbor)
                    stack.append((new_path, depth + 1))
                    
    return None, nodes_explored

for l in [3, 5]:
    path, count = depth_limited_search(agent, 'itching', 'fatigue', l)
    print(f"Limit {l}: {'Path Found!' if path else 'No Path Found within limit.'}")
    if path: print(f"Path: {' -> '.join(path)}")
    print(f"Nodes Explored: {count}\n")

Starting DLS (Limit=3) from 'itching' to 'fatigue'...
Limit 3: Path Found!
Path: itching -> fatigue
Nodes Explored: 142

Starting DLS (Limit=5) from 'itching' to 'fatigue'...
Limit 5: Path Found!
Path: itching -> fatigue
Nodes Explored: 220



In [54]:
#IDS
def iterative_deepening_search(agent, start_node, goal_node, max_depth):
    total_nodes_overall = 0
    
    for limit in range(max_depth + 1):
        print(f"Searching at Depth Limit: {limit} ")
        
        path, nodes_in_this_run = depth_limited_search(agent, start_node, goal_node, limit)

        total_nodes_overall += nodes_in_this_run
        
        if path:
            print(f"SUCCESS: Goal found at depth {limit}!")
            return path, total_nodes_overall
            
        print(f"Goal not found at depth {limit}. Increasing limit...")
        
    return None, total_nodes_overall

start_symptom = 'itching'
target_symptom = 'fatigue'

ids_path, ids_total_count = iterative_deepening_search(agent, start_symptom, target_symptom, 10)
print("\n")
print(f"IDS Result Path: {ids_path}")
print(f"Total Nodes Explored: {ids_total_count}")

Searching at Depth Limit: 0 
Starting DLS (Limit=0) from 'itching' to 'fatigue'...
Goal not found at depth 0. Increasing limit...
Searching at Depth Limit: 1 
Starting DLS (Limit=1) from 'itching' to 'fatigue'...
SUCCESS: Goal found at depth 1!


IDS Result Path: ['itching', 'fatigue']
Total Nodes Explored: 12


In [56]:
#UCS
import heapq
import itertools

counter = itertools.count()  

def uniform_cost_search(agent, start_node, goal_node, severity_map):
    pq = [(0, next(counter), [start_node])] 
    visited = {}
    nodes_explored = 0
    print(f"Starting UCS from '{start_node}' to '{goal_node}'")
    
    while pq:
        current_cost, _, path = heapq.heappop(pq)
        current_symptom = path[-1]
        
        if current_symptom in visited and visited[current_symptom] <= current_cost:
            continue
        
        visited[current_symptom] = current_cost
        nodes_explored += 1
        
        if agent.goal_test(current_symptom, goal_node):
            return path, current_cost, nodes_explored
        
        for neighbor in agent.perceive(current_symptom):
            weight = severity_map.get(neighbor, 1)
            new_cost = current_cost + weight
            if neighbor not in visited or new_cost < visited[neighbor]:
                heapq.heappush(pq, (new_cost, next(counter), path + [neighbor]))
    
    return None, 0, nodes_explored

ucs_path, ucs_total_cost, ucs_nodes = uniform_cost_search(agent, 'itching', 'fatigue', severity_dict)
print(f"\nUCS Result Path: {ucs_path}")
print(f"Total Cumulative Severity: {ucs_total_cost}")
print(f"Nodes Explored: {ucs_nodes}")

Starting UCS from 'itching' to 'fatigue'

UCS Result Path: ['itching', 'fatigue']
Total Cumulative Severity: 4
Nodes Explored: 15


### Uninformed Search Comparison Table

| Algorithm | Path Found | Path Length | Nodes Explored | Optimal? |
| :--- | :--- | :--- | :--- | :--- |
| **BFS** | ['itching', 'fatigue'] | 2 | 17 | Yes |
| **DFS** | ['itching', 'yellowing_of_eyes', 'muscle_pain', 'diarrhoea', 'abdominal_pain', 'coma', 'fatigue'] | 7 | 7 | No |
| **Depth Limited (limit=3)** | ['itching', 'fatigue'] | 2 | 142 | No |
| **Depth Limited (limit=5)** | ['itching', 'fatigue'] | 2 | 220 | No |
| **Iterative Deepening** | ['itching', 'fatigue'] | 2 | 12 | Yes |
| **Uniform Cost Search** | ['itching', 'fatigue'] | 2 | 15 | Yes |


### STEP#3: Informed Searches


In [57]:
import heapq
def heuristic(current_node, goal_node, graph):
    if current_node == goal_node:
        return 0
    neighbors_current = set(graph.get(current_node, [])) 
    neighbors_goal = set(graph.get(goal_node, [])) 
    common = neighbors_current.intersection(neighbors_goal)     
    return 100 - len(common)

In [66]:
#best first search
import heapq
import itertools
bfs_counter = itertools.count()
def best_first_search(agent, start_node, goal_node):
    pq = [(heuristic(start_node, goal_node, agent.graph), next(bfs_counter), [start_node])]
    visited = set()
    nodes_explored = 0  
    print(f"Starting Best-First Search from '{start_node}' to '{goal_node}'")
    while pq:
        h_val, _, path = heapq.heappop(pq)
        current_symptom = path[-1]
        
        if current_symptom in visited:
            continue
            
        visited.add(current_symptom)
        nodes_explored += 1
        
        if agent.goal_test(current_symptom, goal_node):
            return path, nodes_explored
        
        for neighbor in agent.perceive(current_symptom):
            if neighbor not in visited:
                h_score = heuristic(neighbor, goal_node, agent.graph)
                heapq.heappush(pq, (h_score, next(bfs_counter), path + [neighbor]))
    
    return None, nodes_explored

bfs_informed_path, bfs_informed_count = best_first_search(agent, 'itching', 'fatigue')

print(f"\nBest-First Search Result Path: {bfs_informed_path}")
print(f"Nodes Explored: {bfs_informed_count}")

Starting Best-First Search from 'itching' to 'fatigue'

Best-First Search Result Path: ['itching', 'fatigue']
Nodes Explored: 2


In [59]:
#A* SEARCH
def a_star_search(agent, start_node, goal_node, severity_map):
    start_h = heuristic(start_node, goal_node, agent.graph)
    pq = [(start_h, 0, [start_node])]
    visited = {} 
    nodes_explored = 0 
    print(f"Starting A* Search from '{start_node}' to '{goal_node}'")
    while pq:
        f_score, g_score, path = heapq.heappop(pq)
        current_symptom = path[-1]     
        if current_symptom in visited and visited[current_symptom] <= f_score:
            continue      
        visited[current_symptom] = f_score
        nodes_explored += 1   
        if agent.goal_test(current_symptom, goal_node): # Goal Test
            return path, g_score, nodes_explored  
        for neighbor in agent.perceive(current_symptom): 
            weight = severity_map.get(neighbor, 1)
            new_g = g_score + weight         
            h_score = heuristic(neighbor, goal_node, agent.graph)    
            new_f = new_g + h_score 
            
            if neighbor not in visited or new_f < visited[neighbor]:
                new_path = list(path)
                new_path.append(neighbor)
                heapq.heappush(pq, (new_f, new_g, new_path))
                
    return None, 0, nodes_explored

astar_path, astar_cost, astar_nodes = a_star_search(agent, 'itching', 'fatigue', severity_dict)

print(f"\nA* Result Path: {astar_path}")
print(f"Total Path Cost (g): {astar_cost}")
print(f"Nodes Explored: {astar_nodes}")

Starting A* Search from 'itching' to 'fatigue'

A* Result Path: ['itching', 'fatigue']
Total Path Cost (g): 4
Nodes Explored: 2


### STEP#4: BEYOND CLASSICAL SEARCH

In [60]:
#HILL CLIMBING
import random
def hill_climbing(agent, severity_map):
    all_symptoms = list(agent.graph.keys()) # 1. Start at a random symptom from the graph
    current_state = random.choice(all_symptoms)
    
    path = [current_state]
    
    while True:
        current_severity = severity_map.get(current_state, 0)
        neighbors = agent.perceive(current_state)
        
        if not neighbors:
            break
        
        best_neighbor = None 
        best_neighbor_severity = current_severity
        
        for neighbor in neighbors:
            n_severity = severity_map.get(neighbor, 0)
            if n_severity > best_neighbor_severity:
                best_neighbor_severity = n_severity
                best_neighbor = neighbor
        
        if best_neighbor is None: 
            break
            
        current_state = best_neighbor
        path.append(current_state)
        
    return current_state, severity_map.get(current_state, 0), path

print("Running Hill Climbing 10 times\n")

global_max_severity = max(severity_dict.values()) 
global_optimum_count = 0

for i in range(1, 11):
    final_state, severity, path = hill_climbing(agent, severity_dict)
    
    is_global = "GLOBAL OPTIMUM" if severity == global_max_severity else "LOCAL OPTIMUM"
    if severity == global_max_severity:
        global_optimum_count += 1
        
    print(f"Run {i}: Ended at '{final_state}' (Severity: {severity}) -> {is_global}")
    print(f"   Path taken: {' -> '.join(path[:5])}...") 

print(f"\nSummary: Reached Global Optimum {global_optimum_count} out of 10 times.")

Running Hill Climbing 10 times

Run 1: Ended at 'high_fever' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: family_history -> high_fever...
Run 2: Ended at 'burning_micturition' (Severity: 6) -> LOCAL OPTIMUM
   Path taken: burning_micturition...
Run 3: Ended at 'enlarged_thyroid' (Severity: 6) -> LOCAL OPTIMUM
   Path taken: weight_gain -> enlarged_thyroid...
Run 4: Ended at 'high_fever' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: toxic_look_(typhos) -> high_fever...
Run 5: Ended at 'high_fever' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: mucoid_sputum -> high_fever...
Run 6: Ended at 'swelling_joints' (Severity: 5) -> LOCAL OPTIMUM
   Path taken: swelling_joints...
Run 7: Ended at 'high_fever' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: itching -> high_fever...
Run 8: Ended at 'enlarged_thyroid' (Severity: 6) -> LOCAL OPTIMUM
   Path taken: weight_gain -> enlarged_thyroid...
Run 9: Ended at 'chest_pain' (Severity: 7) -> GLOBAL OPTIMUM
   Path taken: chest_pain...
Run 10: End

In [61]:
#SIMULATED ANNEALING
import math
import random

def simulated_annealing(agent, severity_map, start_temp=100, cooling_rate=0.95):
    all_symptoms = list(agent.graph.keys())
    current_state = random.choice(all_symptoms)
    current_severity = severity_map.get(current_state, 0)
    
    temp = start_temp
    path = [current_state]

    while temp > 0.01: 
        neighbors = agent.perceive(current_state)
        if not neighbors:
            break
            
        next_state = random.choice(neighbors)  
        next_severity = severity_map.get(next_state, 0)
        
        delta = next_severity - current_severity 
        
        if delta > 0: 
            current_state = next_state
            current_severity = next_severity
            path.append(current_state)
        else:
            probability = math.exp(delta / temp)
            if random.random() < probability:
                current_state = next_state
                current_severity = next_severity
                path.append(current_state)
        
        temp *= cooling_rate  
        
    return current_state, current_severity, path

print("Running Simulated Annealing")
sa_state, sa_severity, sa_path = simulated_annealing(agent, severity_dict)

print(f"Final State: {sa_state}")
print(f"Final Severity: {sa_severity}")
print(f"Steps taken: {len(sa_path)}")

Running Simulated Annealing
Final State: enlarged_thyroid
Final Severity: 6
Steps taken: 94


In [62]:
#LOCAL BEAM SEARCH
def local_beam_search(agent, severity_map, k):
    all_symptoms = list(agent.graph.keys())
    current_states = random.sample(all_symptoms, k)  
    
    steps = 0
    while True:
        all_neighbors = []
        
        for state in current_states: 
            neighbors = agent.perceive(state)
            for n in neighbors:
                all_neighbors.append((severity_map.get(n, 0), n))  
        
        all_neighbors.sort(key=lambda x: x[0], reverse=True)  
        top_k_neighbors = all_neighbors[:k]
        
        current_best_severity = max([severity_map.get(s, 0) for s in current_states]) 
        new_best_severity = top_k_neighbors[0][0]
        
        if new_best_severity <= current_best_severity:
            break 
            
        current_states = [item[1] for item in top_k_neighbors] 
        steps += 1
        
    best_final_state = current_states[0]
    return best_final_state, severity_map.get(best_final_state, 0), steps

for k_val in [3, 5]:
    state, sev, iterations = local_beam_search(agent, severity_dict, k_val)
    print(f"Beam Search (k={k_val}):")
    print(f"   Best Symptom: '{state}' (Severity: {sev})")
    print(f"   Iterations: {iterations}\n")

Beam Search (k=3):
   Best Symptom: 'high_fever' (Severity: 7)
   Iterations: 1

Beam Search (k=5):
   Best Symptom: 'silver_like_dusting' (Severity: 2)
   Iterations: 0



### STEP#5: ADVERSARIAL SEARCH

In [63]:
#MINIMAX
import math

def minimax(state, depth, is_max_player, agent, severity_map, count_dict):
    count_dict['minimax_nodes'] += 1 

    if depth == 0:
        return severity_map.get(state, 1)

    neighbors = agent.perceive(state)
    if not neighbors:
        return severity_map.get(state, 1)

    if is_max_player:
        best_val = -math.inf
        for neighbor in neighbors:
            value = minimax(neighbor, depth - 1, False, agent, severity_map, count_dict)
            best_val = max(best_val, value)
        return best_val
    else:
        best_val = math.inf
        for neighbor in neighbors:
            value = minimax(neighbor, depth - 1, True, agent, severity_map, count_dict)
            best_val = min(best_val, value)
        return best_val

counters = {'minimax_nodes': 0}
start_symptom = 'itching'
minimax_score = minimax(start_symptom, 4, True, agent, severity_dict, counters)

print(f"Minimax:")
print(f"Start Symptom: {start_symptom}")
print(f"Depth: 4")
print(f"Final Evaluation Score: {minimax_score}")
print(f"Nodes Evaluated: {counters['minimax_nodes']}")

Minimax:
Start Symptom: itching
Depth: 4
Final Evaluation Score: 1
Nodes Evaluated: 486943


In [64]:
#ALPHA BETA PRUNING
def alpha_beta(state, depth, alpha, beta, is_max_player, agent, severity_map, count_dict):
    count_dict['ab_nodes'] += 1
    
    if depth == 0:
        return severity_map.get(state, 1)

    neighbors = agent.perceive(state)
    if not neighbors:
        return severity_map.get(state, 1)

    if is_max_player:
        best_val = -math.inf
        for neighbor in neighbors:
            value = alpha_beta(neighbor, depth - 1, alpha, beta, False, agent, severity_map, count_dict)
            best_val = max(best_val, value)
            alpha = max(alpha, best_val)
            if beta <= alpha:
                break 
        return best_val
    else:
        best_val = math.inf
        for neighbor in neighbors:
            value = alpha_beta(neighbor, depth - 1, alpha, beta, True, agent, severity_map, count_dict)
            best_val = min(best_val, value)
            beta = min(beta, best_val)
            if beta <= alpha:
                break 
        return best_val

counters['ab_nodes'] = 0
ab_score = alpha_beta(start_symptom, 4, -math.inf, math.inf, True, agent, severity_dict, counters)

print(f"Alpha-Beta Pruning:")
print(f"Start Symptom: {start_symptom}")
print(f"Final Evaluation Score: {ab_score}")
print(f"Nodes Evaluated: {counters['ab_nodes']}")
print(f"Nodes Saved (Pruned): {counters['minimax_nodes'] - counters['ab_nodes']}")

Alpha-Beta Pruning:
Start Symptom: itching
Final Evaluation Score: 1
Nodes Evaluated: 27180
Nodes Saved (Pruned): 459763


### PHASE 2 REFLECTION:
**1. Which uninformed search algorithm explored the fewest nodes? Why?**

Iterative Deepening Search (IDS) explored the fewest nodes (12 nodes). This is because the goal fatigue was reachable at depth 1, so IDS stopped very early. DFS explored only 7 nodes but found a non-optimal path (length 7), so it doesn't count as a fair winner. Among algorithms that found the optimal path, IDS was the most efficient.

**2. Did your heuristic improve the search in A compared to UCS? How do you measure improvement?**

Yes, but only slightly in this case. Both UCS and A* explored 2 nodes and found the same path ['itching', 'fatigue'] with the same cost of 4. The heuristic didn't reduce nodes here because the goal was a direct neighbor of the start. Improvement is measured by comparing nodes explored — if A* explores fewer nodes than UCS for the same solution, the heuristic is effective. In general, A* is better on longer, more complex paths.

**3. How often did Hill Climbing get stuck in a local optimum? Did Simulated Annealing help?**

Hill Climbing got stuck in a local optimum 4 out of 10 times (Runs 2, 3, 6, 8), reaching the global optimum 6 out of 10 times. Simulated Annealing ended at enlarged_thyroid with severity 6 in this run, which is a local optimum. However, SA still performs better on average because it can probabilistically escape local traps by accepting worse moves at high temperatures — something Hill Climbing cannot do at all.

**4. How many nodes did Alpha-Beta Pruning eliminate compared to plain Minimax?**

Alpha-Beta Pruning evaluated 27,180 nodes compared to Minimax's 486,943 nodes, eliminating 459,763 nodes — a reduction of approximately 94.4%. This is significantly better than the 50–80% range stated, showing how effective pruning is at depth 4 on a densely connected symptom graph.
